# 00 · Data preparation (sole data egress)

> **This notebook is the project's sole data pipeline.** All `fig*.ipynb` notebooks only read `work/prepared/` and `results/notebooks/00_data/` produced here for statistics and plotting; they do not reprocess raw data.

**Data source**: Stanford HIVDB **complete unfiltered** genotype–phenotype datasets (`{CLASS}_DataSet.Full.txt`). Versus the high-quality filtered set, the Full set is larger and carries metadata such as official `Subtype` and patient `PtID`, enabling true subtype OOD and patient-level leakage control. **No sequence quality filtering is applied; low-quality sequences are recorded as-is but fully retained.**

Recorded and reproducible four-step workflow:

1. **Download** — HIVDB Full datasets + HXB2 reference FASTA (UniProt).
2. **Rebuild** — Overlay position columns (P1..Pn) onto the HXB2 reference to reconstruct full-length protein sequences (self-contained; no external module).
3. **Label** — fold-change ≥ 3.0 → resistant (1), otherwise susceptible (0); missing values remain NaN.
4. **QC & sample caliber** — deduplication, nonstandard amino-acid checks, length checks, and explicit sample caliber for unified downstream use.

**Five drug classes, two groups**

| Group | Drug class | Gene | Use |
|---|---|---|---|
| Main analysis | PI / NRTI / NNRTI / INI | PR / RT / RT / IN | Subtype OOD + patient anti-leakage |
| Extended | CAI | CA | Coverage / cluster OOD only (Subtype is only B/Unknown; no concrete non-B) |


**Sample caliber (must be unified downstream)**

| Caliber | Meaning | Use |
|---|---|---|
| records | Full-set row count | Raw data scale |
| patients | Unique PtID count | Patient-level anti-leakage grouping |
| isolates | Unique SeqID count | Basic modeling/statistics unit |
| unique sequences | Unique rebuilt protein sequences | Sequence redundancy / leakage analysis |

## 0 · Environment and paths

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "work" / "hivdb_full").exists() else CWD.parent
assert (PROJECT_ROOT / "work" / "hivdb_full").exists(), f"cannot locate work/hivdb_full from {CWD}"

# Raw data: HIVDB Full unfiltered set (sole raw-data directory for this project)
RAW_DIR = PROJECT_ROOT / "work" / "hivdb_full"      # {CLASS}_DataSet.Full.txt
REF_DIR = RAW_DIR / "refs"                          # HXB2 reference FASTA (UniProt)
# Outputs
PREP_DIR = PROJECT_ROOT / "work" / "prepared"       # per-class clean tables + summary.json
OUT_DIR = PROJECT_ROOT / "results" / "notebooks" / "00_data"  # summary tables (downstream figures read here)
PREP_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# print("project root:", PROJECT_ROOT)
print("raw dir     :", RAW_DIR.relative_to(PROJECT_ROOT))
print("ref dir     :", REF_DIR.relative_to(PROJECT_ROOT))
print("raw files   :", sorted(p.name for p in RAW_DIR.glob("*_DataSet.Full.txt")))

raw dir     : work/hivdb_full
ref dir     : work/hivdb_full/refs
raw files   : ['CAI_DataSet.Full.txt', 'INI_DataSet.Full.txt', 'NNRTI_DataSet.Full.txt', 'NRTI_DataSet.Full.txt', 'PI_DataSet.Full.txt']


## 1 · Download

All raw inputs live under `work/hivdb_full/`, all from public databases, fully reproducible from scratch.

- **HIVDB Full datasets** `{CLASS}_DataSet.Full.txt` (`{CLASS}` ∈ PI/NRTI/NNRTI/INI/CAI)
  - URL: `https://hivdb.stanford.edu/download/GenoPhenoDatasets/{CLASS}_DataSet.Full.txt` (data version updated 2026-04-22)
  - Contains `SeqID / PtID / Subtype` (official subtype) / per-drug fold-change / `P1..Pn` position columns / `CompMutList`.
  - Versus the "high-quality filtered set" (`{CLASS}_DataSet.txt`): Full is unfiltered, larger, and retains metadata such as Subtype/PtID.
- **HXB2 reference FASTA** (UniProt)
  - `P04585` (Gag-Pol, contains PR/RT/IN, SV=4), `P04591` (Gag, contains CA, SV=3)
  - URL: `https://rest.uniprot.org/uniprotkb/{ACC}.fasta`; HIV-1 group M subtype B isolate HXB2; underlying genome GenBank K03455.

The code block below is **idempotent**: skip download if files are cached; then define task configuration (which drug classes/genes) and inventory inputs. Uses only public HIVDB + UniProt; no third-party scripts or derived annotations.

In [2]:
import urllib.request

# ---- Task config: which drug classes, genes, drugs ----
PAPER_DRUGS = {
    "PI":    ["FPV", "ATV", "IDV", "LPV", "NFV", "SQV", "TPV", "DRV"],
    "NRTI":  ["3TC", "ABC", "AZT", "D4T", "DDI", "TDF"],
    "NNRTI": ["EFV", "ETR", "NVP", "RPV"],
    "INI":   ["RAL", "EVG", "DTG", "BIC", "CAB"],
    "CAI":   ["LEN"],
}
GENE_OF_CLASS = {"PI": "PR", "NRTI": "RT", "NNRTI": "RT", "INI": "IN", "CAI": "CA"}
FILE_OF_CLASS = {cls: RAW_DIR / f"{cls}_DataSet.Full.txt" for cls in PAPER_DRUGS}
REF_ACC = {"P04585": REF_DIR / "P04585.fasta", "P04591": REF_DIR / "P04591.fasta"}
# Main analysis = four classes with usable non-B samples for subtype OOD; extended = CAI (Subtype only B/Unknown)
MAIN_CLASSES = ["PI", "NRTI", "NNRTI", "INI"]
EXT_CLASSES = ["CAI"]
CLASS_ORDER = MAIN_CLASSES + EXT_CLASSES
FC_RESISTANT = 3.0  # clinical threshold used in the paper

# ---- Idempotent download: skip if present ----
HIVDB_URL = "https://hivdb.stanford.edu/download/GenoPhenoDatasets/{cls}_DataSet.Full.txt"
UNIPROT_URL = "https://rest.uniprot.org/uniprotkb/{acc}.fasta"

def fetch(url: str, dest: Path):
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  skip (cached): {dest.name}"); return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  downloading: {dest.name} ...")
    urllib.request.urlretrieve(url, dest)

print("HIVDB Full datasets:")
for cls in CLASS_ORDER:
    fetch(HIVDB_URL.format(cls=cls), FILE_OF_CLASS[cls])
print("UniProt references:")
for acc, dest in REF_ACC.items():
    fetch(UNIPROT_URL.format(acc=acc), dest)

# ---- Inventory (counts only; do not print row contents) ----
inv = []
for cls in CLASS_ORDER:
    raw = pd.read_csv(FILE_OF_CLASS[cls], sep="\t", dtype=str)
    inv.append({"drug_class": cls, "gene": GENE_OF_CLASS[cls],
                "raw_records": len(raw), "n_columns": raw.shape[1],
                "has_subtype": "Subtype" in raw.columns,
                "group": "main" if cls in MAIN_CLASSES else "extended"})
inv = pd.DataFrame(inv)
print("Full records total:", inv["raw_records"].sum(),
      "| main analysis:", inv[inv.group == "main"]["raw_records"].sum(),
      "| extended:", inv[inv.group == "extended"]["raw_records"].sum())
inv


HIVDB Full datasets:
  skip (cached): PI_DataSet.Full.txt
  skip (cached): NRTI_DataSet.Full.txt
  skip (cached): NNRTI_DataSet.Full.txt
  skip (cached): INI_DataSet.Full.txt
  skip (cached): CAI_DataSet.Full.txt
UniProt references:
  skip (cached): P04585.fasta
  skip (cached): P04591.fasta


Full records 合计: 12069 | 主分析: 11830 | 扩展: 239


,drug_class,gene,raw_records,n_columns,has_subtype,group
0,PI,PR,4350,116,True,main
1,NRTI,RT,2652,575,True,main
2,NNRTI,RT,2812,574,True,main
3,INI,IN,2016,302,True,main
4,CAI,CA,239,241,True,extended


## 2 · Sequence rebuild

Build per-gene HXB2 references by slicing two UniProt FASTA files under `refs/`, then patch a few subtype-B consensus polymorphic sites (HIVDB position numbering is relative to these references):

Example:
- reference PR: `PQITLWQRPL VTIKIGGQLK ...` (99 aa; position 10 is L)
- rebuilt: `PQITLWQRPV VTIKIGGQLK ...` (position 10 → V)

| Gene | Source | Polyprotein coordinates (1-based) | Length | Patch sites (→ consensus B) |
|---|---|---|---|---|
| PR | P04585 Gag-Pol | 489–587 | 99 | 3→I |
| RT | P04585 Gag-Pol | 588–1187 | 600 | 214→F, 570→E |
| IN | P04585 Gag-Pol | 1148–1435 | 288 | 10→E, 72→I, 123→S, 124→T, 127→K, 232→D |
| CA | P04591 Gag | 133–363 | 231 | none |

For each row, overlay position columns `P1..Pn` onto the reference to rebuild full-length protein: missing/`-`/`.`/empty → reference (WT); `~` (deletion)/`#` (insertion) → drop that site; letter → take the first amino acid uppercased. This module outputs only rebuilt sequences + metadata (`SeqID/PtID/Subtype/SeqType`), **no labels** (labels are in module 3).


In [3]:
# ---- Build HXB2 references (FASTA slice + consensus-B patches) ----
def read_fasta_seq(path: Path) -> str:
    return "".join(l.strip() for l in path.read_text().splitlines()
                   if l and not l.startswith(">"))

POL = read_fasta_seq(REF_DIR / "P04585.fasta")   # Gag-Pol: contains PR/RT/IN
GAG = read_fasta_seq(REF_DIR / "P04591.fasta")   # Gag: contains CA
REF_SPEC = {  # (source, start, end [1-based inclusive], {protein position: consensus-B AA})
    "PR": (POL, 489, 587, {3: "I"}),
    "RT": (POL, 588, 1187, {214: "F", 570: "E"}),
    "IN": (POL, 1148, 1435, {10: "E", 72: "I", 123: "S", 124: "T", 127: "K", 232: "D"}),
    "CA": (GAG, 133, 363, {}),
}

def build_reference(gene: str) -> str:
    src, start, end, patch = REF_SPEC[gene]
    seq = list(src[start - 1:end])
    for pos, aa in patch.items():
        seq[pos - 1] = aa
    return "".join(seq)

REFERENCE_BY_GENE = {g: build_reference(g) for g in REF_SPEC}

# ---- Self-contained sequence rebuild ----
GAP_CHARS, DEL_CHARS, INS_CHARS = {"-", ".", ""}, {"~"}, {"#"}

def get_position_columns(df: pd.DataFrame) -> list[str]:
    cols = [c for c in df.columns if c.startswith("P") and c[1:].isdigit()]
    return sorted(cols, key=lambda c: int(c[1:]))

def reconstruct_one(row, position_cols, reference) -> str:
    residues = []
    for i, col in enumerate(position_cols):
        aa = row[col]
        ref_aa = reference[i] if i < len(reference) else "X"
        if pd.isna(aa):
            residues.append(ref_aa); continue
        aa = str(aa).strip()
        if aa in GAP_CHARS:
            residues.append(ref_aa)
        elif aa in DEL_CHARS or aa in INS_CHARS:
            continue
        elif aa[0].isalpha():
            residues.append(aa[0].upper())
        else:
            residues.append(ref_aa)
    return "".join(residues)

# Per-class rebuild: produce sequences + metadata; also cache the original df (module 3 uses fold-change)
raw_frames, seq_frames = {}, {}
for cls in CLASS_ORDER:
    df = pd.read_csv(FILE_OF_CLASS[cls], sep="\t", dtype=str)
    gene = GENE_OF_CLASS[cls]
    pos_cols = get_position_columns(df)
    if not pos_cols:
        raise ValueError(f"{cls}: P1..Pn position columns not found")
    ref = REFERENCE_BY_GENE[gene]
    out = pd.DataFrame({"SeqID": df["SeqID"].values})
    for m in ["PtID", "Subtype", "SeqType"]:
        out[m] = df[m].values if m in df.columns else pd.NA
    out["gene"] = gene
    out["sequence"] = [reconstruct_one(r, pos_cols, ref) for _, r in df.iterrows()]
    out["seq_len"] = out["sequence"].str.len()
    raw_frames[cls], seq_frames[cls] = df, out
    print(f"{cls}: reconstructed {len(out)} seqs (gene {gene}, len_mode {int(out.seq_len.mode().iloc[0])})")

PI: reconstructed 4350 seqs (gene PR, len_mode 99)


NRTI: reconstructed 2652 seqs (gene RT, len_mode 560)


NNRTI: reconstructed 2812 seqs (gene RT, len_mode 560)


INI: reconstructed 2016 seqs (gene IN, len_mode 288)
CAI: reconstructed 239 seqs (gene CA, len_mode 231)


## 3 · Labels

For each drug, convert phenotypic fold-change to a binary resistance label: FC ≥ 3.0 → 1 (resistant), < 3.0 → 0 (susceptible), no valid FC → NaN (exclude sample for that drug task).

Merge label columns into module-2 rebuilt sequences and write per-row tables `work/prepared/{CLASS}.csv` (`SeqID, PtID, Subtype, SeqType, gene, sequence, seq_len, <drug>_label...`, one row per raw record, no dedup), plus per-class/per-drug stats in `summary.json`.

In [4]:
# Labels: fold-change -> binary; merge into rebuilt sequences and save
all_stats, frames = {}, {}
for cls in CLASS_ORDER:
    df, out = raw_frames[cls], seq_frames[cls].copy()
    label_stats = {}
    for drug in PAPER_DRUGS[cls]:
        if drug not in df.columns:
            print(f"  [warn] {drug} not in {cls}; skip"); continue
        fc = pd.to_numeric(df[drug], errors="coerce")
        label = fc.apply(lambda v: np.nan if pd.isna(v) else float(v >= FC_RESISTANT))
        out[f"{drug}_label"] = label.values
        n_valid = int(label.notna().sum()); n_res = int((label == 1).sum())
        label_stats[drug] = {"n_valid": n_valid, "n_resistant": n_res,
                             "n_susceptible": n_valid - n_res,
                             "prevalence": round(n_res / n_valid, 4) if n_valid else None}

    n_nonB = int((out["Subtype"].notna() & (out["Subtype"] != "B")).sum())
    all_stats[cls] = {"drug_class": cls, "gene": GENE_OF_CLASS[cls], "n_records": int(len(out)),
                      "n_isolates": int(out["SeqID"].nunique()),
                      "n_patients": int(out["PtID"].nunique()) if out["PtID"].notna().any() else None,
                      "n_unique_sequences": int(out["sequence"].nunique()),
                      "n_nonB": n_nonB, "n_subtypes": int(out["Subtype"].nunique()),
                      "seq_len_mode": int(out["seq_len"].mode().iloc[0]), "drugs": label_stats}
    frames[cls] = out
    out.to_csv(PREP_DIR / f"{cls}.csv", index=False)
    print(f"{cls}: records={len(out)}, isolates={all_stats[cls]['n_isolates']}, "
          f"patients={all_stats[cls]['n_patients']}, non-B={n_nonB}")

(PREP_DIR / "summary.json").write_text(
    json.dumps(all_stats, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved work/prepared/*.csv + summary.json")

PI: records=4350, isolates=4292, patients=3448, non-B=252


NRTI: records=2652, isolates=1600, patients=1391, non-B=96
NNRTI: records=2812, isolates=2732, patients=2098, non-B=160


INI: records=2016, isolates=2016, patients=724, non-B=244
CAI: records=239, isolates=200, patients=19, non-B=49
saved work/prepared/*.csv + summary.json


## 4 · QC and sample caliber

QC the unfiltered Full set and export a unified sample caliber for downstream use so numbers stay consistent across text/figures. **QC only records issues and does not drop low-quality sequences** (pure unfiltered principle).

- **Four-layer sample caliber**: records → patients (unique PtID) → isolates (unique SeqID) → unique sequences (unique rebuilt sequences).
- **Sequence QC**: length consistency, nonstandard amino-acid rate, exact duplicate sequence count.
- **Subtype composition**: use official HIVDB `Subtype`, collapse to B / non-B / Unknown, and keep non-B subtype detail.

produced by `results/notebooks/00_data/`：`sample_caliber.csv`、`sequence_qc.csv`、`data_summary.csv`、`subtype_composition.csv`、`subtype_detail.csv`。

In [5]:
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def collapse_subtype(s):
    if pd.isna(s) or s in ("Unknown", "U", ""):
        return "Unknown"
    return "B" if s == "B" else "non-B"

cal_rows, qc_rows, drug_rows, comp_rows, detail_rows = [], [], [], [], []
for cls in CLASS_ORDER:
    df = frames[cls]
    dedup = df.drop_duplicates(subset=["SeqID"])
    seqs = dedup["sequence"].astype(str)
    lengths = seqs.str.len()
    nonstd = seqs.apply(lambda s: sum(ch not in STANDARD_AA for ch in s))

    # Four-layer caliber
    cal_rows.append({"drug_class": cls, "records": len(df),
                     "patients": int(dedup["PtID"].nunique()) if dedup["PtID"].notna().any() else None,
                     "isolates": dedup["SeqID"].nunique(),
                     "unique_sequences": seqs.nunique(),
                     "records_per_unique_seq": round(len(df) / seqs.nunique(), 3)})
    # Sequence QC
    qc_rows.append({"drug_class": cls, "gene": GENE_OF_CLASS[cls],
                    "len_mode": int(lengths.mode().iloc[0]),
                    "n_len_outliers": int((lengths != lengths.mode().iloc[0]).sum()),
                    "n_seq_nonstd_aa": int((nonstd > 0).sum()),
                    "pct_seq_nonstd_aa": round(100 * (nonstd > 0).mean(), 2),
                    "n_exact_duplicate_seq": int(seqs.duplicated().sum())})
    # Per-drug sample size
    for drug in PAPER_DRUGS[cls]:
        d = all_stats[cls]["drugs"].get(drug)
        if d:
            drug_rows.append({"drug_class": cls, "drug": drug, "gene": GENE_OF_CLASS[cls], **d})
    # Subtype composition + non-B detail
    st = dedup["Subtype"]
    grp = st.map(collapse_subtype).value_counts()
    b, nb, unk = int(grp.get("B", 0)), int(grp.get("non-B", 0)), int(grp.get("Unknown", 0))
    tot = b + nb + unk
    comp_rows.append({"drug_class": cls, "B": b, "non_B": nb, "Unknown": unk,
                      "total": tot, "non_B_pct": round(100 * nb / tot, 1) if tot else 0})
    for sub, n in st[st.map(collapse_subtype) == "non-B"].value_counts().items():
        detail_rows.append({"drug_class": cls, "subtype": sub, "n": int(n)})

def add_all(dfx, sum_cols):
    row = {c: (dfx[c].sum() if c in sum_cols else "ALL") for c in dfx.columns}
    return pd.concat([dfx, pd.DataFrame([row])], ignore_index=True)

caliber = add_all(pd.DataFrame(cal_rows), ["records", "patients", "isolates", "unique_sequences"])
caliber.loc[caliber.index[-1], "records_per_unique_seq"] = round(
    caliber["records"][:-1].sum() / caliber["unique_sequences"][:-1].sum(), 3)
comp = add_all(pd.DataFrame(comp_rows), ["B", "non_B", "Unknown", "total"])
comp.loc[comp.index[-1], "non_B_pct"] = round(
    100 * comp["non_B"][:-1].sum() / comp["total"][:-1].sum(), 1)

caliber.to_csv(OUT_DIR / "sample_caliber.csv", index=False)
pd.DataFrame(qc_rows).to_csv(OUT_DIR / "sequence_qc.csv", index=False)
pd.DataFrame(drug_rows).to_csv(OUT_DIR / "data_summary.csv", index=False)
comp.to_csv(OUT_DIR / "subtype_composition.csv", index=False)
pd.DataFrame(detail_rows).to_csv(OUT_DIR / "subtype_detail.csv", index=False)
print("Four-layer caliber (unfiltered Full set):")
print(caliber.to_string(index=False))
print("\nSubtype composition (official HIVDB field; B-dominant, non-B scarce):")
print(comp.to_string(index=False))

四层口径（未过滤 Full 集）：
drug_class  records  patients  isolates  unique_sequences records_per_unique_seq
        PI     4350      3448      4292              3845                  1.131
      NRTI     2652      1391      1600              1570                  1.689
     NNRTI     2812      2098      2732              2519                  1.116
       INI     2016       724      2016              1410                   1.43
       CAI      239        19       200                89                  2.685
       ALL    12069      7680     10840              9433                  1.279

亚型构成（HIVDB 官方字段，B 占多数、非B稀缺）：
drug_class     B  non_B  Unknown  total non_B_pct
        PI  4040    242       10   4292       5.6
      NRTI  1542     55        3   1600       3.4
     NNRTI  2572    147       13   2732       5.4
       INI  1772    141      103   2016       7.0
       CAI   164      0       36    200       0.0
       ALL 10090    585      165  10840       5.4


## 4b · Global prevalence baseline (external literature for "training vs real-world" contrast)

Global HIV-1 subtype prevalence below comes from a published epidemiological systematic review (**public facts, not third-party code/derived data**), used as a baseline against this project's HIVDB training subtype mix to quantify the clinical pain point that "reference data are B-biased".

- Source: Hemelaar et al. *Global and regional genetic diversity of HIV-1 in 2010–21: systematic review and analysis of prevalence.* **The Lancet Microbe (2024)**. DOI: 10.1016/S2666-5247(24)00151-4
- Window: **2016–2021** (latest period in the paper); 1,044 datasets from 122 countries covering 653,013 PLHIV, weighted by UNAIDS national infection counts.
- Global share (with 95% CI): **C 50.4%**, A 12.4%, **B 11.3%**, CRFs 15.1%, G 2.9%, D 2.6%, URFs 2.0%, F 0.9%, H/J/K each ≤0.1%.
- Numbers are hard-coded from the paper abstract with attribution; downstream figures read `global_subtype_baseline.csv`.

Writes `results/notebooks/00_data/global_subtype_baseline.csv` and `train_vs_global_subtype.csv` (training vs global prevalence).

In [6]:
# ---- Global prevalence baseline (Lancet Microbe 2024; 2016-2021 global % + 95% CI) ----
# Citation: Hemelaar et al., Lancet Microbe 2024, DOI 10.1016/S2666-5247(24)00151-4
# Numbers from paper abstract Findings (2016-2021 window)
GLOBAL_SUBTYPE_2016_21 = [
    # subtype, pct, ci_low, ci_high
    ("C",     50.4, 50.2, 50.7),
    ("A",     12.4, 12.2, 12.6),
    ("B",     11.3, 11.1, 11.5),
    ("G",      2.9,  2.9,  3.0),
    ("D",      2.6,  2.5,  2.7),
    ("F",      0.9,  0.8,  0.9),
    ("CRFs",  15.1, 14.9, 15.3),
    ("URFs",   2.0,  1.9,  2.1),
]
gb = pd.DataFrame(GLOBAL_SUBTYPE_2016_21,
                  columns=["subtype", "global_pct", "ci_low", "ci_high"])
gb["source"] = "Hemelaar_LancetMicrobe_2024"
gb["window"] = "2016-2021"
gb.to_csv(OUT_DIR / "global_subtype_baseline.csv", index=False)

# ---- Training vs global prevalence: B / non-B contrast ----
all_comp = comp[comp["drug_class"] == "ALL"].iloc[0]
train_B = 100 * all_comp["B"] / (all_comp["B"] + all_comp["non_B"])  # B share among known subtypes
global_B = 11.3  # paper 2016-21 global B share
cmp2 = pd.DataFrame({
    "group": ["B", "non-B"],
    "hivdb_train_pct": [round(train_B, 1), round(100 - train_B, 1)],
    "global_2016_21_pct": [global_B, round(100 - global_B, 1)],
})
cmp2.to_csv(OUT_DIR / "train_vs_global_subtype.csv", index=False)
print("Global prevalence baseline (Lancet Microbe 2024, 2016-2021):")
print(gb.to_string(index=False))
print("\nTraining vs global prevalence (B / non-B, %):")
print(cmp2.to_string(index=False))
print(f"\nMismatch: global B is only {global_B}%, but HIVDB training B is {train_B:.1f}%")

全球流行基线（Lancet Microbe 2024, 2016-2021）：
subtype  global_pct  ci_low  ci_high                      source    window
      C        50.4    50.2     50.7 Hemelaar_LancetMicrobe_2024 2016-2021
      A        12.4    12.2     12.6 Hemelaar_LancetMicrobe_2024 2016-2021
      B        11.3    11.1     11.5 Hemelaar_LancetMicrobe_2024 2016-2021
      G         2.9     2.9      3.0 Hemelaar_LancetMicrobe_2024 2016-2021
      D         2.6     2.5      2.7 Hemelaar_LancetMicrobe_2024 2016-2021
      F         0.9     0.8      0.9 Hemelaar_LancetMicrobe_2024 2016-2021
   CRFs        15.1    14.9     15.3 Hemelaar_LancetMicrobe_2024 2016-2021
   URFs         2.0     1.9      2.1 Hemelaar_LancetMicrobe_2024 2016-2021

训练数据 vs 全球流行（B / non-B，%）：
group  hivdb_train_pct  global_2016_21_pct
    B             94.5                11.3
non-B              5.5                88.7

错位：全球 B 仅 11.3%，HIVDB 训练数据 B 却占 94.5%


## Artifact inventory (sole data entry point for downstream figure notebooks)

**Per-row data tables** — `work/prepared/`
- `{PI,NRTI,NNRTI,INI,CAI}.csv`: `SeqID, PtID, Subtype, SeqType, gene, sequence, seq_len, <drug>_label...`
- `summary.json`: per-class n, patients, non-B count, sequence length, per-drug label distribution

**Summary tables** — `results/notebooks/00_data/`
- `sample_caliber.csv`: caliber (records / patients / isolates / unique sequences)
- `data_summary.csv`: per-drug n_valid / n_resistant / prevalence
- `sequence_qc.csv`: length, nonstandard AA, exact duplicates (markedly higher in the unfiltered set)
- `subtype_composition.csv`: class × B/non-B/Unknown composition (official HIVDB subtype)
- `subtype_detail.csv`: non-B subtype detail (C/CRF02_AG/G/...)

> **Contract**: raw data come only from `work/hivdb_full/` (public HIVDB Full set); this notebook builds HXB2 references, implements sequence rebuild, and uses official HIVDB subtypes directly—no third-party code or derived labels. `fig*.ipynb` only `read_csv` the artifacts above.